In [9]:
from pathlib import Path
import os

BASE_DIR = Path(__vsc_ipynb_file__).resolve().parent

Dataset = BASE_DIR / "dataset"

print("CAT TEST DATASET", len(os.listdir(Dataset / "train" / "cat")))  # 5153
print("DOG TEST DATASET", len(os.listdir(Dataset / "train" / "dog")))  # 4739
print("WILD TEST DATASET", len(os.listdir(Dataset / "train" / "wild")))  # 4738

print("CAT VAL DATASET", len(os.listdir(Dataset / "val" / "cat")))
print("DOG VAL DATASET", len(os.listdir(Dataset / "val" / "dog")))
print(
    "WILD VAL DATASET", len(os.listdir(Dataset / "val" / "wild"))
)  # the validation data is equal

CAT TEST DATASET 5153
DOG TEST DATASET 4739
WILD TEST DATASET 4738
CAT VAL DATASET 500
DOG VAL DATASET 500
WILD VAL DATASET 500


In [ ]:
import torch.nn as nn
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
# to feed the data to model in batches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
train_ds = datasets.ImageFolder(Dataset / "train")
print(train_ds.classes)
print(train_ds.class_to_idx)
print(len(train_ds))

['cat', 'dog', 'wild']
{'cat': 0, 'dog': 1, 'wild': 2}
14630


In [16]:
# using compose to chaing multiple transform operation on the images
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),  # to resize the image
        transforms.RandomHorizontalFlip(),  # to flip the image randomly from left to righ
        transforms.RandomRotation(
            10
        ),  # to rotate the image by a random angle (-10,+10)
        transforms.ToTensor(),
    ]
)
val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),  # to resize the image
        transforms.ToTensor(),
    ]
)

train_ds = datasets.ImageFolder(Dataset / "train", transform=train_transform)
val_ds = datasets.ImageFolder(Dataset / "val", transform=val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
num_classes = len(train_ds.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
model = model.to(device)